# 01 - Load, audit and clean the BTS data

**Inputs (raw, downloaded from BTS TranStats - see README):**
* `data/raw/otp/` - Reporting Carrier On-Time Performance, one file per month of each analysed year (2024 and 2025 by default; see `OTP_YEARS` in `scripts/config.py`)
* `data/raw/t100_segment/` - T-100 Domestic Segment (All Carriers), one file per year, 2022 to the latest published month (`T100_FIRST_YEAR`, `T100_LAST_YEAR`)

**Outputs:** cleaned flight table, route-month table (RQ2), carrier-airport table (RQ3),
route monthly passenger series (RQ4), RQ1 modelling sample, a cleaning log with counts.

Every cleaning step is *measured first* (how many rows are affected), then applied,
then logged, so the report can state counts and percentages.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from config import *
from results_store import save_result, load_results
from labels import MONTH_NAMES, DOW_NAMES, carrier_name, airport_name
from bts_io import read_otp, read_t100
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

## 1. Optional: download the On-Time files
Run once from a terminal on a computer that can reach transtats.bts.gov
(`python scripts/download_bts_otp.py`). If a monthly link fails, download that
month from the BTS web form instead and drop the file in `data/raw/otp/`.

## 2. Load the On-Time Performance files

In [2]:
otp, otp_inv = read_otp(RAW_OTP, years=OTP_YEARS)
months_present = sorted(f"{int(y)}-{int(m):02d}" for y, m in otp[["year", "month"]].drop_duplicates().itertuples(index=False))
print(f"\n{len(otp):,} rows x {otp.shape[1]} columns | {len(months_present)} months present: {months_present[0]} to {months_present[-1]}")
missing_months = [f"{y}-{m:02d}" for y in OTP_YEARS for m in range(1, 13) if f"{y}-{m:02d}" not in months_present]
if missing_months:
    print("WARNING: missing months -", missing_months, "- download them (see README) or change OTP_YEARS in scripts/config.py.")
otp_inv.to_csv(TAB / "inventory_otp.csv", index=False)
otp_inv

  read On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip: 547,271 rows, kept 547,271


  read On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_10.zip: 615,497 rows, kept 615,497


  read On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_11.zip: 575,404 rows, kept 575,404


  read On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip: 590,581 rows, kept 590,581


  read On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_2.zip: 519,221 rows, kept 519,221


...

  read On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_7.zip: 631,428 rows, kept 631,428


  read On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_8.zip: 602,378 rows, kept 602,378


  read On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_9.zip: 562,439 rows, kept 562,439



14,080,680 rows x 31 columns | 24 months present: 2024-01 to 2025-12


,file,rows_in_file,rows_kept,periods
0,On_Time_Reporting_Carrier_On_Time_Performance_...,547271,547271,[202401]
1,On_Time_Reporting_Carrier_On_Time_Performance_...,615497,615497,[202410]
2,On_Time_Reporting_Carrier_On_Time_Performance_...,575404,575404,[202411]
3,On_Time_Reporting_Carrier_On_Time_Performance_...,590581,590581,[202412]
4,On_Time_Reporting_Carrier_On_Time_Performance_...,519221,519221,[202402]
5,On_Time_Reporting_Carrier_On_Time_Performance_...,591767,591767,[202403]
6,On_Time_Reporting_Carrier_On_Time_Performance_...,582185,582185,[202404]
7,On_Time_Reporting_Carrier_On_Time_Performance_...,609743,609743,[202405]
8,On_Time_Reporting_Carrier_On_Time_Performance_...,611132,611132,[202406]
9,On_Time_Reporting_Carrier_On_Time_Performance_...,634613,634613,[202407]


In [3]:
# A small UNMODIFIED sample of the raw rows is committed to GitHub so reviewers can inspect the
# data without downloading ~7 million rows (values untouched; column subset only).
raw_sample = otp.sample(min(100_000, len(otp)), random_state=RANDOM_STATE).sort_values("flight_date")
raw_sample.to_csv(RAW_SAMPLE / "otp_raw_sample_100k.csv.gz", index=False)
otp.head(5)

,year,month,day_of_month,day_of_week,flight_date,carrier,carrier_id,flight_number,origin_id,origin,origin_state,dest_id,dest,dest_state,crs_dep_time,dep_delay,dep_del15,dep_time_blk,crs_arr_time,arr_delay,arr_del15,cancelled,cancellation_code,diverted,crs_elapsed,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,1,8,1,2024-01-08,9E,20363,4801.0,12953,LGA,NY,13871,OMA,NE,856,-5.0,0.0,0800-0859,1135,-11.0,0.0,0.0,NaN,0.0,219.0,1148.0,NaN,NaN,NaN,NaN,NaN
1,2024,1,9,2,2024-01-09,9E,20363,4801.0,12953,LGA,NY,13871,OMA,NE,856,-5.0,0.0,0800-0859,1135,-28.0,0.0,0.0,NaN,0.0,219.0,1148.0,NaN,NaN,NaN,NaN,NaN
2,2024,1,10,3,2024-01-10,9E,20363,4801.0,12953,LGA,NY,13871,OMA,NE,856,-6.0,0.0,0800-0859,1135,-25.0,0.0,0.0,NaN,0.0,219.0,1148.0,NaN,NaN,NaN,NaN,NaN
3,2024,1,11,4,2024-01-11,9E,20363,4801.0,12953,LGA,NY,13871,OMA,NE,856,23.0,1.0,0800-0859,1135,27.0,1.0,0.0,NaN,0.0,219.0,1148.0,0.0,0.0,4.0,0.0,23.0
4,2024,1,12,5,2024-01-12,9E,20363,4801.0,12953,LGA,NY,13871,OMA,NE,856,-5.0,0.0,0800-0859,1135,2.0,0.0,0.0,NaN,0.0,219.0,1148.0,NaN,NaN,NaN,NaN,NaN


## 3. Audit BEFORE cleaning
Each check below measures a data-quality issue. Nothing is changed in this section.

In [4]:
audit = {"raw_rows": int(len(otp)), "raw_columns_used": int(otp.shape[1])}
n_raw = len(otp)

# 3a. missing values -------------------------------------------------------------------------
miss = otp.isna().sum().rename("missing").to_frame()
miss["pct"] = (miss["missing"] / n_raw * 100).round(3)
miss = miss.sort_values("missing", ascending=False)
miss.to_csv(TAB / "missingness_otp_raw.csv")

canc = otp["cancelled"].fillna(0).eq(1)
div = otp["diverted"].fillna(0).eq(1)
audit["cancelled"], audit["diverted"] = int(canc.sum()), int(div.sum())
arr_missing = otp["arr_delay"].isna()
audit["arr_delay_missing_total"] = int(arr_missing.sum())
audit["arr_delay_missing_explained"] = int((arr_missing & (canc | div)).sum())      # cancelled / diverted
audit["arr_delay_missing_unexplained"] = int((arr_missing & ~canc & ~div).sum())    # true data gaps
cause_cols = [c for c in ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"] if c in otp.columns]
late_flag = otp["arr_del15"].eq(1)
audit["late_flights"] = int(late_flag.sum())
audit["cause_missing_among_late"] = int(otp.loc[late_flag, cause_cols].isna().all(axis=1).sum()) if cause_cols else None
miss.head(12)

,missing,pct
cancellation_code,13881489,98.585
late_aircraft_delay,11096078,78.804
security_delay,11096078,78.804
nas_delay,11096078,78.804
weather_delay,11096078,78.804
carrier_delay,11096078,78.804
arr_del15,235949,1.676
arr_delay,235949,1.676
dep_del15,191561,1.360
dep_delay,191561,1.360


In [5]:
# 3b. duplicates -----------------------------------------------------------------------------
dup_key = [k for k in ["flight_date", "carrier_id", "flight_number", "origin_id", "dest_id", "crs_dep_time"] if k in otp.columns]
dup = otp.duplicated(subset=dup_key, keep="first")
audit["duplicates"] = int(dup.sum())

# 3c. range / logic checks ---------------------------------------------------------------------
crs = pd.to_numeric(otp["crs_dep_time"], errors="coerce")
bad_distance = ~(otp["distance"] > 0)
same_od = otp["origin_id"] == otp["dest_id"]
bad_time = ~crs.between(0, 2400)
op_mask = otp["arr_del15"].notna() & otp["arr_delay"].notna()
inconsistent = op_mask & ((otp["arr_delay"] >= 15).astype(float) != otp["arr_del15"].astype(float))
audit.update({"bad_distance": int(bad_distance.sum()), "origin_equals_dest": int(same_od.sum()),
              "bad_sched_time": int(bad_time.sum()), "arrdel15_inconsistent": int(inconsistent.sum())})

# 3d. outliers in arrival delay (Tukey 'extreme' fences = 3 x IQR) -----------------------------
ad = otp.loc[op_mask, "arr_delay"].astype(float)
q1, q3 = ad.quantile([0.25, 0.75]); iqr = q3 - q1
hi, lo = q3 + 3 * iqr, q1 - 3 * iqr
audit.update({"arr_delay_q1": float(q1), "arr_delay_q3": float(q3), "arr_delay_hi_fence": float(hi),
              "arr_delay_lo_fence": float(lo), "outliers_high": int((ad > hi).sum()), "outliers_low": int((ad < lo).sum()),
              "arr_delay_max": float(ad.max()), "arr_delay_min": float(ad.min()),
              "arr_delay_p999": float(ad.quantile(0.999)), "arr_delay_over_600": int((ad > 600).sum()),
              "operated_rows_raw": int(op_mask.sum())})
{k: audit[k] for k in audit}

{'raw_rows': 14080680,
 'raw_columns_used': 31,
 'cancelled': 199191,
 'diverted': 36757,
 'arr_delay_missing_total': 235949,
 'arr_delay_missing_explained': 235948,
 'arr_delay_missing_unexplained': 1,
 'late_flights': 2984604,
 'cause_missing_among_late': 2,
 'duplicates': 0,
 'bad_distance': 0,
 'origin_equals_dest': 0,
 'bad_sched_time': 0,
 'arrdel15_inconsistent': 0,
 'arr_delay_q1': -15.0,
 'arr_delay_q3': 10.0,
 'arr_delay_hi_fence': 85.0,
 'arr_delay_lo_fence': -90.0,
 'outliers_high': 707202,
 'outliers_low': 11,
 'arr_delay_max': 4336.0,
 'arr_delay_min': -128.0,
 'arr_delay_p999': 753.0,
 'arr_delay_over_600': 20417,
 'operated_rows_raw': 13844731}

## 4. Apply the cleaning decisions
1. **Duplicates** - drop repeated flight records (same date, carrier, flight number, origin, destination, scheduled time).
2. **Invalid records** - drop rows with non-positive distance or origin = destination (not real flights).
3. **Cancelled / diverted flights** - *kept* (RQ3 needs cancellations) but flagged; they have no arrival delay, so they are excluded from delay analyses by an `operated` flag.
4. **Missing delay-cause minutes** - structurally missing (BTS fills them only for delays of 15+ minutes); kept as missing, treated as 0 only inside late flights for the EDA.
5. **Outliers** - extreme arrival delays are real events (e.g., long ground stops), so they are **retained**. Robust statistics (median, rank correlation) and clipped *display* ranges are used instead of deleting data.

In [6]:
valid = ~dup & ~bad_distance & ~same_od
audit["removed_duplicates"] = int(dup.sum())
audit["removed_invalid"] = int((~dup & (bad_distance | same_od)).sum())
flights = otp.loc[valid].copy()
del otp

flights["cancelled"] = flights["cancelled"].fillna(0).astype("int8")
flights["diverted"] = flights["diverted"].fillna(0).astype("int8")
flights["operated"] = ((flights["cancelled"] == 0) & (flights["diverted"] == 0) & flights["arr_del15"].notna()).astype("int8")
flights["is_late"] = flights["arr_del15"].astype("float32")          # NaN when not operated
crs = pd.to_numeric(flights["crs_dep_time"], errors="coerce")
flights["dep_hour"] = ((crs // 100) % 24).astype("Int8")
for c in ("carrier_id", "origin_id", "dest_id"):
    flights[c] = flights[c].astype("int64")

audit["clean_rows"] = int(len(flights))
audit["clean_operated"] = int(flights["operated"].sum())
audit["clean_late_rate_pct"] = float(flights.loc[flights.operated == 1, "is_late"].mean() * 100)
audit["clean_cancel_rate_pct"] = float(flights["cancelled"].mean() * 100)
flights.to_parquet(PROCESSED / "otp_clean.parquet", index=False)
flights[["flight_date", "carrier", "origin", "dest", "dep_time_blk", "distance", "arr_delay", "is_late", "cancelled"]].head(8).to_csv(TAB / "otp_clean_preview.csv", index=False)
print(f"clean rows: {audit['clean_rows']:,}  operated: {audit['clean_operated']:,}  "
      f"late rate: {audit['clean_late_rate_pct']:.2f}%  cancel rate: {audit['clean_cancel_rate_pct']:.2f}%")

clean rows: 14,080,680  operated: 13,844,731  late rate: 21.56%  cancel rate: 1.41%


## 5. T-100 Domestic Segment (capacity and passengers)

In [7]:
t100, t100_inv = read_t100(RAW_T100, T100_FIRST_YEAR, T100_LAST_YEAR)
t_audit = {"raw_rows": int(len(t100))}
t_audit["service_class_counts"] = {str(k): int(v) for k, v in t100["service_class"].value_counts().items()}
t100 = t100[t100["service_class"] == SERVICE_CLASS].copy()
t_audit["class_f_rows"] = int(len(t100))
for c in ["seats", "passengers", "dep_performed", "dep_scheduled", "distance"]:
    if c in t100.columns:
        t100[c] = pd.to_numeric(t100[c], errors="coerce")
if "dep_scheduled" not in t100.columns:
    t100["dep_scheduled"] = np.nan
# a few BTS rows have no carrier / airport ID, so they cannot be joined to the On-Time data: count, then drop
id_cols = ["carrier_id", "origin_id", "dest_id", "year", "month"]
for c in id_cols:
    t100[c] = pd.to_numeric(t100[c], errors="coerce")
miss_id = t100[id_cols].isna().any(axis=1)
t_audit["missing_key_rows"] = int(miss_id.sum())
t_audit["missing_key_passengers"] = float(t100.loc[miss_id, "passengers"].sum())
if miss_id.any():
    print("rows without a carrier/airport ID (dropped):", int(miss_id.sum()), "by year:", t100.loc[miss_id, "year"].value_counts(dropna=False).to_dict())
t100 = t100[~miss_id].copy()
for c in id_cols:
    t100[c] = t100[c].astype("int64")
# rows are split by aircraft type, so aircraft columns are part of the duplicate check when present
dup_t = t100.duplicated(keep="first")
t_audit["duplicates"] = int(dup_t.sum())
t100 = t100[~dup_t]
t_audit["missing_cells"] = {c: int(t100[c].isna().sum()) for c in ["seats", "passengers", "dep_performed", "distance"]}
t100_inv.to_csv(TAB / "inventory_t100.csv", index=False)
print(t_audit)
t100_inv

  read t100_segment_2022.csv: 411,573 rows, kept 411,573


  read t100_segment_2023.csv: 427,427 rows, kept 427,427


  read t100_segment_2024.csv: 440,939 rows, kept 440,939


  read t100_segment_2025.csv: 458,793 rows, kept 458,793


  read t100_segment_2026.csv: 268,069 rows, kept 268,069


rows without a carrier/airport ID (dropped): 24 by year: {2022: 24}


{'raw_rows': 2006801, 'service_class_counts': {'F': 1532260, 'L': 260201, 'G': 162352, 'P': 51988}, 'class_f_rows': 1532260, 'missing_key_rows': 24, 'missing_key_passengers': 3249.0, 'duplicates': 0, 'missing_cells': {'seats': 0, 'passengers': 0, 'dep_performed': 0, 'distance': 0}}


,file,rows_in_file,rows_kept,years
0,t100_segment_2022.csv,411573,411573,[2022]
1,t100_segment_2023.csv,427427,427427,[2023]
2,t100_segment_2024.csv,440939,440939,[2024]
3,t100_segment_2025.csv,458793,458793,[2025]
4,t100_segment_2026.csv,268069,268069,[2026]


In [8]:
# aggregate aircraft-type rows to one row per carrier x segment x month
keys = ["year", "month", "carrier_id", "origin_id", "dest_id"]
t_rm = (t100.groupby(keys, as_index=False)
        .agg(passengers=("passengers", "sum"), seats=("seats", "sum"), dep_performed=("dep_performed", "sum"),
             dep_scheduled=("dep_scheduled", "sum"), distance=("distance", "mean"),
             origin=("origin", "first"), dest=("dest", "first")))
t_rm["load_factor"] = 100 * t_rm["passengers"] / t_rm["seats"].where(t_rm["seats"] > 0)
t_audit["segment_months"] = int(len(t_rm))
t_audit["lf_missing_or_zero_seats"] = int(t_rm["load_factor"].isna().sum())
t_audit["lf_over_100"] = int((t_rm["load_factor"] > 100).sum())
t_audit["lf_median"] = float(t_rm["load_factor"].median())
t_rm.loc[t_rm["load_factor"] > 100, "load_factor"] = np.nan   # impossible values -> treated as invalid for load-factor analyses
t_rm.head()

,year,month,carrier_id,origin_id,dest_id,passengers,seats,dep_performed,dep_scheduled,distance,origin,dest,load_factor
0,2022,1,19393,10140,10423,3667.0,4610.0,30.0,31.0,619.0,ABQ,AUS,79.544469
1,2022,1,19393,10140,10800,1063.0,2145.0,15.0,15.0,672.0,ABQ,BUR,49.557110
2,2022,1,19393,10140,10821,1293.0,1622.0,10.0,10.0,1670.0,ABQ,BWI,79.716400
3,2022,1,19393,10140,11259,10375.0,15194.0,102.0,107.0,580.0,ABQ,DAL,68.283533
4,2022,1,19393,10140,11292,6568.0,11736.0,72.0,83.0,349.0,ABQ,DEN,55.964554


## 6. Join the two tables and build the analysis tables
Join keys are the stable DOT-assigned IDs (`carrier_id`, `origin_id`, `dest_id`) plus `year` and `month`, never airport/carrier codes, because codes can be reused.

In [9]:
otp_rm = (flights.groupby(["year", "month", "carrier_id", "origin_id", "dest_id"], observed=True, as_index=False)
          .agg(flights_scheduled=("flight_date", "size"), flights_operated=("operated", "sum"),
               late=("is_late", "sum"), cancelled=("cancelled", "sum"), distance_otp=("distance", "mean"),
               carrier=("carrier", "first"), origin=("origin", "first"), dest=("dest", "first")))
t_y = t_rm[t_rm["year"].isin(OTP_YEARS)].drop(columns=["origin", "dest"])
for c in ("year", "month"):
    otp_rm[c] = otp_rm[c].astype("int64")
rm = otp_rm.merge(t_y, on=["year", "month", "carrier_id", "origin_id", "dest_id"], how="left", indicator=True)
matched = rm["_merge"].eq("both")
join = {"otp_route_months": int(len(rm)), "matched_route_months": int(matched.sum()),
        "matched_share_route_months_pct": float(matched.mean() * 100),
        "matched_share_flights_pct": float(rm.loc[matched, "flights_scheduled"].sum() / rm["flights_scheduled"].sum() * 100)}
print(join)

enough = rm["flights_operated"] >= MIN_FLIGHTS_ROUTE_MONTH
lf_ok = rm["load_factor"].between(0.0001, 100)
join["excluded_unmatched"] = int((~matched).sum())
join["excluded_too_few_flights"] = int((matched & ~enough).sum())
join["excluded_bad_load_factor"] = int((matched & enough & ~lf_ok).sum())
rma = rm[matched & enough & lf_ok].copy()
rma["late_rate"] = 100 * rma["late"] / rma["flights_operated"]
rma["cancelled"] = rma["cancelled"].astype("int64")   # sums can come back as a small integer type, and 100 * a small integer overflows
rma["cancel_rate"] = 100 * rma["cancelled"] / rma["flights_scheduled"]
rma = rma.drop(columns="_merge")
rma.to_csv(PROCESSED / "route_month_analysis.csv", index=False)
join["rq2_route_months"] = int(len(rma))
join["rq2_routes"] = int(rma[["carrier_id", "origin_id", "dest_id"]].drop_duplicates().shape[0])
print(join)
rma.head()

{'otp_route_months': 234892, 'matched_route_months': 234796, 'matched_share_route_months_pct': 99.95913015343221, 'matched_share_flights_pct': 99.99924719544795}


{'otp_route_months': 234892, 'matched_route_months': 234796, 'matched_share_route_months_pct': 99.95913015343221, 'matched_share_flights_pct': 99.99924719544795, 'excluded_unmatched': 96, 'excluded_too_few_flights': 86555, 'excluded_bad_load_factor': 0, 'rq2_route_months': 148241, 'rq2_routes': 9998}


,year,month,carrier_id,origin_id,dest_id,flights_scheduled,flights_operated,late,cancelled,distance_otp,carrier,origin,dest,passengers,seats,dep_performed,dep_scheduled,distance,load_factor,late_rate,cancel_rate
0,2024,1,19393,10140,10423,49,48,15.0,1,619.0,WN,ABQ,AUS,5252.0,7120.0,48.0,49.0,619.0,73.764045,31.250000,2.040816
2,2024,1,19393,10140,10821,31,30,11.0,1,1670.0,WN,ABQ,BWI,3219.0,4930.0,30.0,31.0,1670.0,65.294118,36.666667,3.225806
3,2024,1,19393,10140,11259,145,140,17.0,5,580.0,WN,ABQ,DAL,14048.0,21140.0,140.0,145.0,580.0,66.452223,12.142857,3.448276
4,2024,1,19393,10140,11292,118,110,17.0,8,349.0,WN,ABQ,DEN,10094.0,17138.0,110.0,118.0,349.0,58.898355,15.454545,6.779661
5,2024,1,19393,10140,12191,74,73,10.0,1,759.0,WN,ABQ,HOU,8307.0,11271.0,73.0,74.0,759.0,73.702422,13.698630,1.351351


In [10]:
# prior-month route load factor (known BEFORE departure) for RQ1 -------------------------------
prev = t_rm[["year", "month", "carrier_id", "origin_id", "dest_id", "load_factor"]].dropna().copy()
nxt = pd.to_datetime(dict(year=prev["year"], month=prev["month"], day=1)) + pd.offsets.MonthBegin(1)
prev["year"], prev["month"] = nxt.dt.year.astype(int), nxt.dt.month.astype(int)
prev = prev.rename(columns={"load_factor": "lf_prev"})

op = flights[flights["operated"] == 1]
n_samp = min(RQ1_SAMPLE_N, len(op))
samp = op.sample(n_samp, random_state=RANDOM_STATE)
samp = samp.merge(prev, on=["year", "month", "carrier_id", "origin_id", "dest_id"], how="left")
keep = ["flight_date", "year", "month", "day_of_week", "dep_time_blk", "dep_hour", "carrier", "carrier_id",
        "origin", "dest", "origin_id", "dest_id", "distance", "crs_elapsed", "lf_prev", "is_late", "arr_delay"]
samp[keep].to_csv(PROCESSED / "rq1_sample.csv.gz", index=False)
rq1_info = {"sample_n": int(len(samp)), "late_rate_pct": float(samp["is_late"].mean() * 100),
            "lf_prev_missing_pct": float(samp["lf_prev"].isna().mean() * 100)}
print(rq1_info)

{'sample_n': 400000, 'late_rate_pct': 21.581748962402344, 'lf_prev_missing_pct': 1.238}


In [11]:
# carrier x origin-airport table for RQ3 ----------------------------------------------------------
t_o = (t_y.assign(w=lambda d: d["distance"] * d["dep_performed"])
       .assign(ym=lambda d: d["year"] * 100 + d["month"])
       .groupby(["carrier_id", "origin_id", "ym"], as_index=False)
       .agg(passengers=("passengers", "sum"), seats=("seats", "sum"), dep=("dep_performed", "sum"), w=("w", "sum")))
g = (t_o.groupby(["carrier_id", "origin_id"])
     .agg(passengers=("passengers", "sum"), seats=("seats", "sum"), dep_total=("dep", "sum"), dep_mean=("dep", "mean"),
          dep_std=("dep", "std"), months=("ym", "nunique"), w=("w", "sum")).reset_index())
g["avg_load_factor"] = 100 * g["passengers"] / g["seats"].where(g["seats"] > 0)
g["avg_distance"] = g["w"] / g["dep_total"].where(g["dep_total"] > 0)
g["seasonal_cv"] = g["dep_std"] / g["dep_mean"]
g = g.rename(columns={"dep_mean": "mean_monthly_departures"})

otp_ca = (flights.groupby(["carrier_id", "origin_id"], observed=True, as_index=False)
          .agg(flights_scheduled=("flight_date", "size"), cancelled=("cancelled", "sum"),
               carrier=("carrier", "first"), origin=("origin", "first")))
ca = otp_ca.merge(g[["carrier_id", "origin_id", "avg_load_factor", "mean_monthly_departures", "avg_distance", "seasonal_cv", "months"]],
                  on=["carrier_id", "origin_id"], how="inner")
ca_info = {"carrier_airport_pairs_otp": int(len(otp_ca)), "pairs_matched_t100": int(len(ca))}
ca = ca[(ca["flights_scheduled"] >= MIN_FLIGHTS_CARRIER_AIRPORT) & (ca["months"] >= MIN_MONTHS_CARRIER_AIRPORT)].dropna(
    subset=["avg_load_factor", "mean_monthly_departures", "avg_distance", "seasonal_cv"])
ca = ca[ca["avg_load_factor"].between(0.0001, 100)]
ca["cancelled"] = ca["cancelled"].astype("int64")
ca["cancel_rate"] = 100 * ca["cancelled"] / ca["flights_scheduled"]
ca.to_csv(PROCESSED / "carrier_airport.csv", index=False)
ca_info["rq3_units"] = int(len(ca)); ca_info["rq3_flights"] = int(ca["flights_scheduled"].sum())
print(ca_info)
ca.head()

{'carrier_airport_pairs_otp': 1814, 'pairs_matched_t100': 1814, 'rq3_units': 1598, 'rq3_flights': 14044433}


,carrier_id,origin_id,flights_scheduled,cancelled,carrier,origin,avg_load_factor,mean_monthly_departures,avg_distance,seasonal_cv,months,cancel_rate
0,19393,10140,23750,141,WN,ABQ,71.969072,990.833333,610.971741,0.099615,24,0.593684
1,19393,10257,7641,72,WN,ALB,82.771299,316.666667,763.323553,0.106495,24,0.942285
2,19393,10279,3778,27,WN,AMA,67.689578,160.708333,421.891626,0.068249,24,0.714664
3,19393,10397,54850,477,WN,ATL,71.918473,2274.208333,643.217603,0.260343,24,0.869644
4,19393,10423,75127,447,WN,AUS,73.858471,3122.375000,851.824066,0.083015,24,0.594992


In [12]:
# monthly route passenger series for RQ4 (all carriers combined, scheduled passenger service) ------
t_r = (t_rm.groupby(["origin_id", "dest_id", "year", "month"], as_index=False)
       .agg(passengers=("passengers", "sum"), origin=("origin", "first"), dest=("dest", "first")))
t_r["period"] = pd.to_datetime(dict(year=t_r["year"], month=t_r["month"], day=1))
rows_per_period = t_r.groupby("period").size()
# drop a trailing, only-partly-published month (rows far below the typical month)
while len(rows_per_period) and rows_per_period.iloc[-1] < 0.8 * rows_per_period.median():
    print("dropping partial month", rows_per_period.index[-1].date())
    rows_per_period = rows_per_period.iloc[:-1]
periods = rows_per_period.index
wide = t_r[t_r["period"].isin(periods)].pivot_table(index="period", columns=["origin_id", "dest_id"], values="passengers", aggfunc="sum")
complete = wide.columns[(wide.notna().all(axis=0)) & (wide.min(axis=0) > 0)]
top = wide[complete].sum().sort_values(ascending=False).head(RQ4_N_ROUTES).index
lab = t_r.drop_duplicates(["origin_id", "dest_id"]).set_index(["origin_id", "dest_id"])[["origin", "dest"]]
series = wide[top].copy()
series.columns = [f"{lab.loc[c, 'origin']}-{lab.loc[c, 'dest']}" for c in series.columns]
series.index.name = "period"
series.to_csv(PROCESSED / "route_monthly_passengers.csv")
rq4_info = {"months": int(len(series)), "first_period": str(series.index.min().date()), "last_period": str(series.index.max().date()),
            "routes_with_complete_series": int(len(complete)), "routes_used": int(series.shape[1])}
print(rq4_info)

{'months': 55, 'first_period': '2022-01-01', 'last_period': '2026-07-01', 'routes_with_complete_series': 5101, 'routes_used': 150}


In [13]:
# airport id -> code lookup (documentation / labelling) ----------------------------------------------
lookup = (pd.concat([flights[["origin_id", "origin", "origin_state"]].rename(columns={"origin_id": "airport_id", "origin": "code", "origin_state": "state"}),
                     flights[["dest_id", "dest", "dest_state"]].rename(columns={"dest_id": "airport_id", "dest": "code", "dest_state": "state"})])
          .astype({"code": "str", "state": "str"}).drop_duplicates("airport_id").sort_values("airport_id"))
lookup.to_csv(PROCESSED / "airport_lookup.csv", index=False)
print(len(lookup), "airports")

358 airports


## 7. Cleaning log and feasibility check

In [14]:
pct = lambda n, d: round(100 * n / d, 3)
n0 = audit["raw_rows"]
log = pd.DataFrame([
    ["Missing values (arrival delay)", "arr_delay, arr_del15", "Count of NaN; cross-tab with cancelled/diverted flags",
     "Kept; excluded from delay analyses via the 'operated' flag", "Missing arrival delay is structural: cancelled and diverted flights never arrive normally",
     audit["arr_delay_missing_total"], pct(audit["arr_delay_missing_total"], n0)],
    ["Missing values (unexplained)", "arr_delay", "NaN where flight was neither cancelled nor diverted",
     "Excluded from delay analyses", "True data gaps cannot be imputed without bias",
     audit["arr_delay_missing_unexplained"], pct(audit["arr_delay_missing_unexplained"], n0)],
    ["Missing values (delay causes)", "carrier/weather/nas/security/late-aircraft delay", "NaN among late flights",
     "Kept missing; treated as 0 minutes only inside late flights (EDA)", "BTS records causes only for delays of 15+ minutes",
     audit["cause_missing_among_late"] if audit["cause_missing_among_late"] is not None else 0, np.nan],
    ["Duplicates", "flight_date, carrier_id, flight_number, origin_id, dest_id, crs_dep_time", "duplicated() on the flight key",
     "Dropped (first kept)", "A flight can occur once per scheduled departure", audit["removed_duplicates"], pct(audit["removed_duplicates"], n0)],
    ["Invalid distance / route", "distance, origin_id, dest_id", "distance <= 0 or origin == destination",
     "Dropped", "Not a real flight leg", audit["removed_invalid"], pct(audit["removed_invalid"], n0)],
    ["Logical inconsistency", "arr_del15 vs arr_delay", "arr_del15 != (arr_delay >= 15)",
     "Reported (no correction needed if 0)", "Confirms the binary flag matches the minutes field", audit["arrdel15_inconsistent"], pct(audit["arrdel15_inconsistent"], n0)],
    ["Outliers (extreme delay)", "arr_delay", "Tukey fence Q3 + 3 x IQR",
     "Retained; robust statistics and clipped display ranges used", "Extreme delays are real operational events, not errors",
     audit["outliers_high"] + audit["outliers_low"], pct(audit["outliers_high"] + audit["outliers_low"], audit["operated_rows_raw"])],
    ["T-100 rows without carrier or airport ID", "carrier_id, origin_id, dest_id", "Missing key field",
     "Dropped (cannot be joined to the On-Time table)", "Stable IDs are the join keys; passengers on these rows are a negligible share",
     t_audit["missing_key_rows"], pct(t_audit["missing_key_rows"], t_audit["class_f_rows"])],
    ["T-100 duplicates", "all fields", "duplicated() on whole row", "Dropped", "Exact repeats are reporting artefacts",
     t_audit["duplicates"], pct(t_audit["duplicates"], t_audit["class_f_rows"])],
    ["T-100 impossible load factor", "passengers, seats", "load factor > 100% after summing aircraft types",
     "Set to missing for load-factor analyses", "Passengers cannot exceed seats offered", t_audit["lf_over_100"], pct(t_audit["lf_over_100"], t_audit["segment_months"])],
], columns=["Issue", "Variables affected", "Detection method", "Treatment applied", "Rationale", "Count", "Percent"])
log.to_csv(TAB / "cleaning_log.csv", index=False)

feas = pd.DataFrame([
    ["RQ1", "flights in modelling sample", rq1_info["sample_n"], load_results()["sample_size"]["rq1"]],
    ["RQ2", "route-months", join["rq2_route_months"], load_results()["sample_size"]["rq2"]],
    ["RQ3", "flights in clustered carrier-airports", ca_info["rq3_flights"], load_results()["sample_size"]["rq3_total"]],
    ["RQ4", "routes with complete series", rq4_info["routes_used"], load_results()["sample_size"]["rq4"]],
], columns=["RQ", "Unit", "Available", "Minimum required"])
feas["Requirement met"] = feas["Available"] >= feas["Minimum required"]
feas.to_csv(TAB / "feasibility_check.csv", index=False)
save_result("data_overview", {"otp_audit": audit, "t100_audit": t_audit, "join": join, "carrier_airport": ca_info,
                              "rq1_sample": rq1_info, "rq4_series": rq4_info, "otp_months": months_present,
                              "otp_columns_total_in_table": 109})
log, feas

(                                      Issue                                 Variables affected                                   Detection method  \
 0            Missing values (arrival delay)                               arr_delay, arr_del15  Count of NaN; cross-tab with cancelled/diverte...   
 1              Missing values (unexplained)                                          arr_delay  NaN where flight was neither cancelled nor div...   
 2             Missing values (delay causes)   carrier/weather/nas/security/late-aircraft delay                             NaN among late flights   
 3                                Duplicates  flight_date, carrier_id, flight_number, origin...                     duplicated() on the flight key   
 4                  Invalid distance / route                       distance, origin_id, dest_id             distance <= 0 or origin == destination   
 5                     Logical inconsistency                             arr_del15 vs arr_delay     

**Reading the result.** The cleaning log (above) is reproduced in the report. The feasibility table
confirms the real data meets every minimum sample size from notebook 00.